# Week 6 — Plotly Visualizations for Mini Project 1

This notebook uses `Week 5/stars.csv` and builds interactive Plotly charts for the three core questions:

1. Which properties best separate star types?
2. Are larger stars always brighter, and does that change by type?
3. Does star color reliably indicate temperature and type?


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

STAR_TYPE_LABELS = {
    0: "Brown Dwarf",
    1: "Red Dwarf",
    2: "White Dwarf",
    3: "Main Sequence",
    4: "Supergiant",
    5: "Hypergiant",
}

STAR_TYPE_ORDER = [
    "Brown Dwarf",
    "Red Dwarf",
    "White Dwarf",
    "Main Sequence",
    "Supergiant",
    "Hypergiant",
]

COLOR_NORMALIZATION = {
    "blue white": "Blue-White",
    "blue-white": "Blue-White",
    "bluewhite": "Blue-White",
    "blue": "Blue",
    "red": "Red",
    "white": "White",
    "white-yellow": "White-Yellow",
    "yellow-white": "Yellow-White",
    "yellowish white": "Yellow-White",
    "yellowish": "Yellowish",
    "yellowish-orange": "Yellowish-Orange",
    "orange": "Orange",
    "orange-red": "Orange-Red",
    "pale yellow orange": "Pale Yellow-Orange",
    "whitish": "Whitish",
}


In [ ]:
def clean_star_color(value: str) -> str:
    key = str(value).strip().lower().replace("  ", " ")
    key = key.replace("_", " ")
    return COLOR_NORMALIZATION.get(key, str(value).strip().title())


DATA_PATH = Path("../Week 5/stars.csv")
df = pd.read_csv(DATA_PATH)

df["Star_type_label"] = df["Star type"].map(STAR_TYPE_LABELS)
df["Star_color_clean"] = df["Star color"].apply(clean_star_color)
df["log10_Luminosity"] = df["Luminosity(L/Lo)"].apply(lambda v: np.log10(v) if v > 0 else np.nan)
df["log10_Radius"] = df["Radius(R/Ro)"].apply(lambda v: np.log10(v) if v > 0 else np.nan)

print("shape:", df.shape)
print("missing values:")
print(df.isna().sum())

df.head()

## Question 1

### Which star properties are most useful for separating star types?

We compare distributions of temperature, luminosity, radius, and absolute magnitude across star types.
Luminosity and radius are visualized on log10-transformed values so all types remain visible.

In [ ]:
metrics = [
    "Temperature (K)",
    "Luminosity(L/Lo)",
    "Radius(R/Ro)",
    "Absolute magnitude(Mv)",
]

long_df = df.melt(
    id_vars=["Star_type_label"],
    value_vars=metrics,
    var_name="Metric",
    value_name="Value",
)

log_metrics = {"Luminosity(L/Lo)", "Radius(R/Ro)"}
long_df["Metric_for_plot"] = long_df["Metric"].where(
    ~long_df["Metric"].isin(log_metrics),
    "log10(" + long_df["Metric"] + ")",
)
long_df["Value_for_plot"] = long_df["Value"]
for metric in log_metrics:
    mask = long_df["Metric"] == metric
    long_df.loc[mask, "Value_for_plot"] = long_df.loc[mask, "Value"].apply(
        lambda v: np.log10(v) if v > 0 else np.nan
    )

fig_q1_box = px.box(
    long_df.dropna(subset=["Value_for_plot"]),
    x="Star_type_label",
    y="Value_for_plot",
    color="Star_type_label",
    facet_col="Metric_for_plot",
    facet_col_wrap=2,
    category_orders={"Star_type_label": STAR_TYPE_ORDER},
    title="Q1: Which properties separate star types?",
    labels={
        "Star_type_label": "Star Type",
        "Value_for_plot": "Value (log where indicated)",
    },
)
fig_q1_box.update_layout(showlegend=False, height=800)
fig_q1_box.show()

q1_means = (
    df.groupby("Star_type_label", observed=True)[metrics]
    .mean()
    .reindex(STAR_TYPE_ORDER)
    .reset_index()
)
q1_means_long = q1_means.melt(
    id_vars=["Star_type_label"], var_name="Metric", value_name="Mean"
)
fig_q1_means = px.bar(
    q1_means_long,
    x="Star_type_label",
    y="Mean",
    color="Metric",
    barmode="group",
    category_orders={"Star_type_label": STAR_TYPE_ORDER},
    title="Q1: Mean values by star type",
)
fig_q1_means.show()

## Question 2

### Are larger stars always brighter, or does the relationship change by star type?

A log-log scatter makes this relationship visible across many orders of magnitude.
A faceted version shows how the relationship differs within each type.

In [ ]:
fig_q2_scatter = px.scatter(
    df,
    x="Radius(R/Ro)",
    y="Luminosity(L/Lo)",
    color="Star_type_label",
    size="Temperature (K)",
    hover_data=[
        "Temperature (K)",
        "Absolute magnitude(Mv)",
        "Star_color_clean",
        "Spectral Class",
    ],
    log_x=True,
    log_y=True,
    category_orders={"Star_type_label": STAR_TYPE_ORDER},
    title="Q2: Are larger stars always brighter?",
)
fig_q2_scatter.show()

fig_q2_facet = px.scatter(
    df,
    x="Radius(R/Ro)",
    y="Luminosity(L/Lo)",
    color="Star_type_label",
    facet_col="Star_type_label",
    facet_col_wrap=3,
    log_x=True,
    log_y=True,
    hover_data=["Temperature (K)", "Absolute magnitude(Mv)", "Star_color_clean"],
    category_orders={"Star_type_label": STAR_TYPE_ORDER},
    title="Q2: Radius vs luminosity by star type",
)
fig_q2_facet.update_layout(showlegend=False, height=800)
fig_q2_facet.show()

## Question 3

### Does a star's color reliably indicate temperature and type?

First, we use cleaned color labels. Then we compare temperature spread by color and inspect type-spectral structure using a heatmap.

In [ ]:
color_order = (
    df.groupby("Star_color_clean", observed=True)["Temperature (K)"]
    .mean()
    .sort_values()
    .index.tolist()
)

fig_q3_color_temp = px.box(
    df,
    x="Star_color_clean",
    y="Temperature (K)",
    color="Star_type_label",
    category_orders={
        "Star_color_clean": color_order,
        "Star_type_label": STAR_TYPE_ORDER,
    },
    title="Q3: Temperature distribution by cleaned star color",
    labels={"Star_color_clean": "Cleaned Star Color", "Temperature (K)": "Temperature (K)"},
)
fig_q3_color_temp.show()

pivot = (
    df.pivot_table(
        index="Star_type_label",
        columns="Spectral Class",
        values="Temperature (K)",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(STAR_TYPE_ORDER)
    .fillna(0)
)

fig_q3_heatmap = go.Figure(
    data=[
        go.Heatmap(
            z=pivot.values,
            x=pivot.columns.tolist(),
            y=pivot.index.tolist(),
            colorbar={"title": "Count"},
        )
    ]
)
fig_q3_heatmap.update_layout(
    title="Q3: Star type x spectral class",
    xaxis_title="Spectral Class",
    yaxis_title="Star Type",
)
fig_q3_heatmap.show()

## Quick takeaway template

- **Q1:** The properties that separate types most clearly are `____` and `____`.
- **Q2:** Radius and luminosity show an overall trend, but `____` types break a simple one-rule explanation.
- **Q3:** Color is useful after cleaning labels, but overlap across colors means it is a `____` indicator rather than a perfect classifier.